# Text-to-SVG Generation: Training Pipeline

**Model:** Qwen2.5-Coder-3B-Instruct with LoRA fine-tuning 
**Task:** Generate valid SVG code from natural language prompts 
**Competition:** NYU DL Spring 2026 Kaggle Contest

## 1. Environment Setup

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET
import multiprocessing

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    multiprocessing.set_start_method('spawn', force=True)
except RuntimeError:
    pass

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Data Preprocessing

We normalize all SVGs to a standard 256×256 canvas, and filter out invalid/oversized samples.

**Key preprocessing steps:**
- Strip XML namespaces
- Validate against allowed SVG tags (Kaggle whitelist)
- Scale coordinates to 256×256 canvas
- Drop SVGs with `currentColor` (no explicit color)
- Drop SVGs with `transform` attributes (complex coordinate transforms)
- Filter by character length (≤2500 chars)
- Minify whitespace


In [ ]:
import re
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm.auto import tqdm

ALLOWED_TAGS = {
    'svg', 'g', 'path', 'rect', 'circle', 'ellipse', 'line', 'polyline', 
    'polygon', 'defs', 'use', 'symbol', 'clipPath', 'mask', 'linearGradient', 
    'radialGradient', 'stop', 'text', 'tspan', 'title', 'desc', 'style', 
    'pattern', 'marker', 'filter'
}

def normalize_complex_svg(svg_str):
    """
    1. Keeps all complex paths and tags (Prevents the Kaggle TED penalty).
    2. Drops 'currentColor' to prevent color pollution.
    3. Natively scales all coordinates to fit perfectly in a 256x256 box.
    """
    if not isinstance(svg_str, str) or len(svg_str) > 16000:
        return None
    if svg_str.count("<path") > 256:
        return None
        
    # Drop samples that don't have explicit colors
    if 'currentColor' in svg_str:
        return None
        
    try:
        # Strip XML namespaces for clean parsing
        svg_str = re.sub(r'\sxmlns=[\'"][^\'"]+[\'"]', '', svg_str)
        svg_str = re.sub(r'\sxmlns:\w+=[\'"][^\'"]+[\'"]', '', svg_str)
        
        root = ET.fromstring(svg_str)
        
        # 1. Check for disallowed Kaggle tags
        for elem in root.iter():
            tag = elem.tag.split('}')[-1]
            if tag not in ALLOWED_TAGS:
                return None
                
        # 2. Calculate the True Scale Factor
        if 'viewBox' in root.attrib:
            viewbox = root.attrib['viewBox'].split()
            orig_width = float(viewbox[2])
        elif 'width' in root.attrib:
            orig_width = float(re.sub(r'[^\d.]', '', root.attrib['width']))
        else:
            orig_width = 256.0 
            
        scale_factor = 256.0 / orig_width

        # Smart rounding: Keeps 1 decimal for smooth curves, drops .0 for integers
        def scale_and_round(match):
            val = float(match.group(0)) * scale_factor
            rounded = round(val, 1)
            if rounded.is_integer():
                return str(int(rounded))
            return str(rounded)

        # 3. Scale all coordinates mathematically
        if abs(scale_factor - 1.0) > 0.01:
            for child in root.iter():
                tag = child.tag.split('}')[-1]
                
                # Scale basic attributes
                for attr in ['x', 'y', 'width', 'height', 'cx', 'cy', 'r', 'rx', 'ry', 'x1', 'y1', 'x2', 'y2']:
                    if attr in child.attrib:
                        raw_val = re.sub(r'[^\d.-]', '', child.attrib[attr])
                        if raw_val:
                            scaled_val = float(raw_val) * scale_factor
                            child.attrib[attr] = str(int(scaled_val)) if scaled_val.is_integer() else str(round(scaled_val, 1))
                            
                # Scale paths
                if tag == 'path' and 'd' in child.attrib:
                    d_attr = child.attrib['d']
                    # Matches floats and ints, applies scale multiplier
                    child.attrib['d'] = re.sub(r'-?\d+(?:\.\d+)?', scale_and_round, d_attr)
                    

        # 4. Extract inner content and wrap in Kaggle strict 256x256 header
        modified_svg_str = ET.tostring(root, encoding='unicode', method='xml')
        inner_content_match = re.search(r'<svg[^>]*>(.*?)</svg>', modified_svg_str, flags=re.IGNORECASE | re.DOTALL)
        
        if not inner_content_match:
            return None
            
        inner_content = inner_content_match.group(1).strip()
        
        standard_header = '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        new_svg = f'{standard_header}{inner_content}</svg>'

        # 5. Minify (Remove tabs, newlines, excess spaces)
        new_svg = new_svg.replace('\n', '').replace('\r', '').replace('\t', '')
        new_svg = re.sub(r'\s+', ' ', new_svg) 
        new_svg = re.sub(r'\s*filling="[^"]*"', '', new_svg)
        new_svg = new_svg.replace('> <', '><') 

        
        return new_svg.strip()
        
    except Exception:
        return None

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    train_df = pd.read_csv("train.csv")
    print(f"Original samples: {len(train_df)}")
    
    tqdm.pandas(desc="Normalizing SVGs")
    train_df['svg'] = train_df['svg'].progress_apply(normalize_complex_svg)
    
    # Drop failed/banned SVGs
    train_df = train_df.dropna(subset=['svg'])
    
    # Keep SVGs that will comfortably fit in max_seq_length=4096 (approx 12,000 characters)
    train_df = train_df[train_df['svg'].str.len() <= 2500]

    # Drop SVGs with transforms to prevent corrupted spatial learning
    train_df = train_df[~train_df['svg'].str.contains('transform', case=False, na=False)]
    
    print(f"Kept complex, scaled samples: {len(train_df)}")
    train_df.to_csv("train_complex.csv", index=False)

## 3. Z-Index Area Sorting

Sort drawable elements by approximate bounding box area (largest first) to enforce a painter's algorithm ordering. This ensures background shapes render behind foreground details.


In [ ]:
import re
import xml.etree.ElementTree as ET

def get_approx_area(elem):
    # Strip namespace if present
    tag = elem.tag.split('}')[-1]
    
    try:
        if tag == 'rect':
            w = float(elem.get('width', 0))
            h = float(elem.get('height', 0))
            return w * h
            
        elif tag == 'circle':
            r = float(elem.get('r', 0))
            return (r * 2) ** 2  # Bounding box area of circle
            
        elif tag == 'ellipse':
            rx = float(elem.get('rx', 0))
            ry = float(elem.get('ry', 0))
            return (rx * 2) * (ry * 2)
            
        elif tag in ['path', 'polygon', 'polyline', 'line']:
            # Combine all possible coordinate attributes
            attr_text = f"{elem.get('d', '')} {elem.get('points', '')} {elem.get('x1', '')} {elem.get('x2', '')} {elem.get('y1', '')} {elem.get('y2', '')}"
            
            # Extract all numbers
            nums = [float(n) for n in re.findall(r'-?\d+(?:\.\d+)?', attr_text)]
            if len(nums) < 2:
                return 0
                
            spread = max(nums) - min(nums)
            return spread ** 2
            
        return 0 # Defs, groups, etc. get pushed to the end or handled separately
    except:
        return 0

def z_index_sort_svg(svg_str):
    try:
        root = ET.fromstring(svg_str)
        
        # Separate structural tags (like <defs>) from drawable tags
        drawables = []
        non_drawables = []
        
        for elem in root:
            if elem.tag.split('}')[-1] in ['path', 'rect', 'circle', 'ellipse', 'polygon', 'polyline', 'line']:
                drawables.append(elem)
            else:
                non_drawables.append(elem)
        
        # Sort ALL drawable shapes descending by their area (Largest first)
        drawables.sort(key=get_approx_area, reverse=True)
        
        # Rebuild the SVG
        for child in list(root): 
            root.remove(child)
            
        root.extend(non_drawables + drawables)
        
        # Return cleanly formatted XML
        return ET.tostring(root, encoding='unicode')
    except Exception as e:
        return svg_str

# Apply to your dataset
train_df['svg'] = train_df['svg'].apply(z_index_sort_svg)

## 4. Training Configuration

**LoRA Hyperparameters:**
- `r=128, alpha=256` → effective scaling = 2.0× (acts as LR multiplier)
- All attention + MLP projections targeted

**Ablation note:** We tested `r=128, alpha=128` (scaling=1.0) with `lr=5e-5`. The effective LR dropped from 2e-4 to 5e-5, causing severe under-training (score dropped from 14.0 to 11.4). The model could not override base model artifacts like `filling="0"`.

In [ ]:
import multiprocessing
try:
    multiprocessing.set_start_method('spawn', force=True)
except RuntimeError:
    pass
CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-3B-Instruct",
    "max_seq_length": 2560,
 
    # --- LoRA ---
    "lora_r": 128,
    "lora_alpha": 256,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
 
    # --- Training ---
    "learning_rate": 1e-4,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
 
    # --- Logging & Saving ---
    "logging_steps": 20,
    "eval_steps": 500, 
    "save_steps": 500,
    "max_train_samples": 50000,
    "eval_size": 0.02,
    "output_dir": "./outputs_long_ver12",
 
}
 

CONFIG

## 5. Data Loading & Splitting

In [ ]:
import pandas as pd
from datasets import Dataset
 
SEED = 42
 
CLEANED_CSV_PATH = "train_complex.csv"
RAW_CSV_PATH = "train.csv"
 
print(f"Loading cleaned dataset from {CLEANED_CSV_PATH}...")
train_df = pd.read_csv(CLEANED_CSV_PATH)
 
# Convert to HuggingFace Dataset
full_ds = Dataset.from_pandas(train_df)
 
if CONFIG["max_train_samples"] < len(full_ds):
    full_ds = full_ds.shuffle(seed=SEED).select(range(CONFIG["max_train_samples"]))
 
# Split into Train and Validation
splits = full_ds.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]
 
 
# Save raw eval for local scoring
print(f"Loading raw dataset from {RAW_CSV_PATH}...")
raw_df = pd.read_csv(RAW_CSV_PATH)
eval_ids = eval_ds['id']
raw_eval_df = raw_df[raw_df['id'].isin(eval_ids)]
raw_eval_df.to_csv("eval.csv", index=False)
 
print(f"Train rows: {len(train_ds)}")
print(f"Eval rows:  {len(raw_eval_df)}")

## 6. System Prompt & SFT Formatting

The system prompt instructs the model on SVG generation constraints. The training data is formatted using the ChatML template (`<|im_start|>` / `<|im_end|>`).

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert vector graphics designer"
    "Generate highly compact, parseable, and valid SVG code"
    "Focus on the dominant colors and bounding boxes of the requested objects"
)

def format_sft_text(example):
    # We directly map the 'prompt' and 'svg' columns from train.csv
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}

# Apply the formatting to the datasets and remove the old columns
train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names, num_proc=8)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print("--- Preview of formatted training data ---")
print(train_text[0]["text"][:500])

## 7. Model Loading & LoRA Setup

We use Unsloth for efficient LoRA fine-tuning on the Qwen2.5-Coder-3B-Instruct base model. The model is loaded in full precision (not quantized) to maximize generation quality within the 4B parameter competition limit.

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from safetensors.torch import load_file

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=False,
)

# 2. PERMANENTLY FIX PADDING FOR DECODER MODELS
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if len(tokenizer) > model.config.vocab_size:
    model.resize_token_embeddings(len(tokenizer))

# 3. Setup the Chat Template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml"
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    # use_rslora=True,
    # use_dora=True,
    lora_dropout=0,
    bias="none",
    target_modules=CONFIG["target_modules"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

## 8. Training

**Training details:**
- Optimizer: AdamW 8-bit
- Scheduler: Cosine with warmup
- Gradient clipping: max_norm=0.3
- Response-only loss masking (only train on SVG output, not on prompt tokens)


In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer
from unsloth.chat_templates import train_on_responses_only
import transformers.trainer
 
transformers.trainer.check_torch_load_is_safe = lambda: None
 
training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
 
    warmup_ratio=CONFIG["warmup_ratio"],
 
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
 
    # --- Eval & Save ---
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=5,
 
    max_grad_norm=0.3, 
 
    report_to="none",
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
    dataloader_num_workers=0,
)
 
 
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,
    args=training_args,
)
 
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
 
# Start training
train_result = trainer.train()

## 9. Save Model

Save the LoRA adapter weights and tokenizer for inference.

In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

## 10. Reproducibility Notes

- **Random seed:** `SEED=42` set for Python, NumPy, and PyTorch
- **Hardware:** NVIDIA RTX 4080 Super (16GB VRAM)
- **Training time:** ~7.5 hours for 3 epochs on 50k samples
- **AI Tooling:** Claude (Anthropic) was used for coding assistance and debugging